In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import wandb
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
    get_canonical_chart_transport_monitor_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.state import MultimodalTrainingRuntime

/tmp/ipykernel_3940560/587479404.py:11: TqdmExperimentalWarning:

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)



In [2]:
num_modes = 8
latent_dimension = 2
ambient_dimension = 2
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=0.1,
    offset=0.0,
    ambient_dimension=ambient_dimension,
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    chart_transport_config=get_canonical_chart_transport_configs(
        data_config=data_config,
        latent_dimension=latent_dimension,
    ),
    monitor_config=get_canonical_chart_transport_monitor_configs(),
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / experiment_name
    ),
)

training_config.visualize()

In [3]:
cuda_device = 1
runtime = MultimodalTrainingRuntime.initialize(
    tc=training_config,
    cuda_device=cuda_device,
    wandb_project=wandb_project,
    run_name=experiment_name,
)

Using bfloat16 Automatic Mixed Precision (AMP)


Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Pretrain

Chart pretrain is now delegated to the runtime. It logs constraint and conditioning monitors on the configured schedule.


In [4]:
latest_pretrain_metrics = runtime.chart_pretrain()
latest_pretrain_metrics

pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[pretrain] step 1/1000: train: chart_loss=0.8034, data_cycle_loss=0.2436, prior_cycle_loss=0.5598, zero_mean_loss=0.0312, latent_norm_loss=0.1734; monitor: constraint_reconstruction_mean=3.5013, constraint_reconstruction_max=5.2336, constraint_latent_norm_mean=3.8878, encoder_conditioning_mean=5.6654, encoder_conditioning_max=423.3269, decoder_conditioning_mean=1.0573, decoder_conditioning_max=4.3777
[pretrain] step 1000/1000: train: chart_loss=0.0002, data_cycle_loss=0.0000, prior_cycle_loss=0.0000, zero_mean_loss=0.1350, latent_norm_loss=1.1011; monitor: constraint_reconstruction_mean=0.0050, constraint_reconstruction_max=0.0238, constraint_latent_norm_mean=1.3310, encoder_conditioning_mean=1.1520, encoder_conditioning_max=5.9099, decoder_conditioning_mean=1.0179, decoder_conditioning_max=4.7937


{'train_chart_loss': 0.00015596555022057146,
 'train_data_cycle_loss': 1.0965572982968297e-05,
 'train_prior_cycle_loss': 3.489214941509999e-05,
 'train_zero_mean_loss': 0.13495635986328125,
 'train_latent_norm_loss': 1.1010782718658447,
 'monitor_constraint_reconstruction_mean': 0.005049602128565311,
 'monitor_constraint_reconstruction_max': 0.02376079186797142,
 'monitor_constraint_latent_norm_mean': 1.3309742212295532,
 'monitor_encoder_conditioning_mean': 1.152024745941162,
 'monitor_encoder_conditioning_max': 5.909936904907227,
 'monitor_decoder_conditioning_mean': 1.0178946256637573,
 'monitor_decoder_conditioning_max': 4.793697357177734}

## Train noise critic

In [ ]:
latest_critic_pretrain_metrics = runtime.critic_pretrain()
latest_critic_pretrain_metrics

critic_pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[critic_pretrain] step 1/1000: train: critic_loss=1.0898; monitor: critic_monitor_snapshot_score_norm_mean=239.9843, critic_monitor_transport_norm_mean=51.9557, critic_monitor_transport_norm_max=67.0755, critic_monitor_transport_t_min=0.0100, critic_monitor_transport_t_max=0.9900
[critic_pretrain] step 1000/1000: train: critic_loss=0.3411; monitor: critic_monitor_snapshot_score_norm_mean=12.2452, critic_monitor_transport_norm_mean=2.5600, critic_monitor_transport_norm_max=7.0603, critic_monitor_transport_t_min=0.0100, critic_monitor_transport_t_max=0.9900


{'train_critic_loss': 0.341128408908844,
 'monitor_critic_monitor_snapshot_score_norm_mean': 12.24518871307373,
 'monitor_critic_monitor_transport_norm_mean': 2.560030698776245,
 'monitor_critic_monitor_transport_norm_max': 7.0602545738220215,
 'monitor_critic_monitor_transport_t_min': 0.009999999776482582,
 'monitor_critic_monitor_transport_t_max': 0.9900000095367432}

: 

## Integrated training

Integrated training is now delegated to the runtime. Each integrated step performs one chart-repair update, `n_critic_updates_every_transport_step` critic updates, then one chart-transport update. The integrated monitor stack logs constraint, critic, conditioning, and sampling monitors on the configured schedule.


In [ ]:
latest_integrated_metrics = runtime.integrated()
latest_integrated_metrics

integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: train: critic=0.3462, repair=0.0001, data_cycle=0.0000, prior_cycle=0.0001, data_dual=0.0000, prior_dual=0.0000, transport=0.0093, enc_transport=0.0047, dec_transport=0.0046, field=2.0393; monitor: recon_err=0.0246, latent_norm=1.2290, score=11.5187, field=3.2885, enc_cond=1.1568, dec_cond=1.0372, kl_0p001=5.0817, kl_0p003=4.7169, kl_0p01=4.6047, kl_0p03=4.5665
[integrated] step 1000/1000000: train: critic=0.4115, repair=0.0012, data_cycle=0.0004, prior_cycle=0.0008, data_dual=0.0000, prior_dual=0.0000, transport=0.0083, enc_transport=0.0046, dec_transport=0.0037, field=1.6834; monitor: recon_err=0.0389, latent_norm=1.0915, score=10.3260, field=2.7927, enc_cond=1.5026, dec_cond=1.0338, kl_0p001=3.9822, kl_0p003=3.8947, kl_0p01=3.8990, kl_0p03=3.9446
[integrated] step 2000/1000000: train: critic=0.3787, repair=0.0012, data_cycle=0.0003, prior_cycle=0.0009, data_dual=0.0000, prior_dual=0.0000, transport=0.0085, enc_transport=0.0043, dec_transport=0.0043, fiel